In [1]:
import math

from scipy.io import arff
from operator import index

import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors, KernelDensity
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import scipy.stats
from scipy.stats import expon, skew, norm,gamma, anderson,goodness_of_fit, monte_carlo_test, probplot, skewnorm
from scipy import integrate
from sklearn.metrics import auc
import seaborn as sns
import math

from statsmodels.sandbox.distributions.gof_new import kstest

plt.rcParams['figure.figsize'] = [15, 7]
import warnings
from scipy import stats

warnings.filterwarnings('ignore')

In [2]:
class ParametricMethod:
    def __init__(self,filename,p,logTrue=False,distribution=stats.gamma):
        self.distance = []
        self.fileName = filename
        self.X = 0
        self.y = 0
        self.arr = []
        self.logTrue = logTrue
        self.p = p
        self.tots = []
        self.distribution = distribution
        self.dataframe = pd.DataFrame()

    def generateOutput(self):
        self._readArff()
        for v in range(2,70):
            self._distanceMetric(v)
            self._generateArray()
            if self.logTrue:
                self.arr = np.log(self.arr)
            params = self.distribution.fit(self.arr) # fit params for gamma distribution
            posNeg4 = []
            spaceStep4 = np.linspace(0,.99,30) # threshold from 0 to .99, 30 samples
            for e in spaceStep4:
                if len(params) == 3:
                    newArr = self.arr > self.distribution.ppf(e,params[0],loc=params[1], scale=params[2]) # if arr value is outside threshold add to new array
                else:
                    newArr = self.arr > self.distribution.ppf(e,loc=params[0], scale=params[1]) # if arr value is outside threshold add to new array
                posNeg4.append([((self.y[newArr] == 1).sum() / (self.y == 1).sum()), (self.y[newArr] != 1).sum()/ ((self.y != 1).sum())]) # True positive rate, false positive rate

            posNeg4 = np.array(posNeg4)
            arrtest1, arrtest2 = np.split(posNeg4, 2,axis=1) # split the array
            self.tots += [auc(arrtest2, arrtest1)] # return the area under the curve

        maxValue, kIndex = self._printResults(self.tots)
        return maxValue, kIndex

    def _distanceMetric(self,n):
        #find the nearestNeighbors
        nn = NearestNeighbors(n_neighbors=n,p=self.p)
        nn.fit(self.X, self.y)
        #return the dist of each and the nearest neighbors
        self.distance, knn = nn.kneighbors(self.X)  # returns N index neighbors including self

    def _readArff(self):
        arff_file = arff.loadarff(f'./{self.fileName}') # import the attribute-relation file format
        df4 = pd.DataFrame(arff_file[0])
        self.X = df4.drop(columns=['outlier','id']).values
        #get outlier values
        self.y = df4['outlier'].values
        le = LabelEncoder()
        #encoded the variables as 0=non-outlier, 1=outlier
        self.y = le.fit_transform(self.y)

    def _generateArray(self):
        self.arr = []
        #returns an array based on the median and max values
        for x in self.distance:  # finds the distance away from that point (index 0)
            self.arr += [np.max(x)]

    def _printResults(self,totalArr):
        newarr = np.nan_to_num(totalArr)
        newarr = list(newarr)
        print(max(newarr),newarr.index(max(newarr))+2) #print the max values, the k value, and the array
        return max(newarr),newarr.index(max(newarr))+2

In [3]:
positively_skewed_distributions = [
    stats.expon,
    stats.chi2,
    stats.gamma,
    stats.weibull_min,
    stats.lognorm,
    stats.invgauss,
    stats.rayleigh,
    stats.wald,
    stats.pareto,
    stats.levy,
    stats.nakagami,
    stats.logistic,
    stats.powerlaw,
    stats.skewnorm
]

folder_structure_1d = [
    "semantic/Annthyroid/Annthyroid_withoutdupl_norm_07.arff",
    "semantic/Arrhythmia/Arrhythmia_withoutdupl_norm_46.arff",
    "semantic/Cardiotocography/Cardiotocography_withoutdupl_norm_22.arff",
    "semantic/HeartDisease/HeartDisease_withoutdupl_norm_44.arff",
    "semantic/Hepatitis/Hepatitis_withoutdupl_norm_16.arff",
    "semantic/InternetAds/InternetAds_withoutdupl_norm_19.arff",
    "semantic/PageBlocks/PageBlocks_withoutdupl_norm_09.arff",
    "semantic/Parkinson/Parkinson_withoutdupl_norm_75.arff",
    "semantic/Pima/Pima_withoutdupl_norm_35.arff",
    "semantic/SpamBase/SpamBase_withoutdupl_norm_40.arff",
    "semantic/Stamps/Stamps_withoutdupl_norm_09.arff",
    "semantic/Wilt/Wilt_withoutdupl_norm_05.arff"
]





In [4]:
dict = {}
for z in folder_structure_1d:
    print(z)
    dict[z] = {}
    for i in positively_skewed_distributions:
        print(i)
        holder = ParametricMethod(z,1,distribution=i)
        maxVal, indexMax = holder.generateOutput()
        dict[z][i] = [maxVal,indexMax]

df = pd.DataFrame.from_dict(dict)

# Print DataFrame


semantic/Annthyroid/Annthyroid_withoutdupl_norm_07.arff
0.6761145800501458 2
0.6763770930764142 2
0.6764974884502787 2
0.6760776663741966 2
0.6758756349862142 2
0.6769205759669252 2
0.6759453450434872 2
0.6763670128033665 2
0.6760085242196305 2
0.6768747178233426 2
0.6762105556076133 2
0.6735908204206454 2
0.6746039588497698 2
0.6751869109784112 2
semantic/Arrhythmia/Arrhythmia_withoutdupl_norm_46.arff
0.7565653350310361 35
0.7611013846888429 46
0.7610814897342033 41
0.7596590004774788 44
0.7599375298424319 45
0.760335428935222 45
0.7598778449785134 45
0.7604846410950183 44
0.7606636956867737 35
0.7610715422568837 37
0.7602459016393441 38
0.761499283781633 29
0.7608725927104887 45
0.7620265000795797 45
semantic/Cardiotocography/Cardiotocography_withoutdupl_norm_22.arff
0.5564958435768157 69
0.5568532803450144 69
0.5571775126046918 69
0.5578806669027877 68
0.5563024761448393 69
0.5564209706654443 67
0.5575811752573024 69
0.5575609921246719 69
0.5564964946456102 69
0.5580538512021334 69


In [5]:
df.head()

,semantic/Annthyroid/Annthyroid_withoutdupl_norm_07.arff,semantic/Arrhythmia/Arrhythmia_withoutdupl_norm_46.arff,semantic/Cardiotocography/Cardiotocography_withoutdupl_norm_22.arff,semantic/HeartDisease/HeartDisease_withoutdupl_norm_44.arff,semantic/Hepatitis/Hepatitis_withoutdupl_norm_16.arff,semantic/InternetAds/InternetAds_withoutdupl_norm_19.arff,semantic/PageBlocks/PageBlocks_withoutdupl_norm_09.arff,semantic/Parkinson/Parkinson_withoutdupl_norm_75.arff,semantic/Pima/Pima_withoutdupl_norm_35.arff,semantic/SpamBase/SpamBase_withoutdupl_norm_40.arff,semantic/Stamps/Stamps_withoutdupl_norm_09.arff,semantic/Wilt/Wilt_withoutdupl_norm_05.arff
<scipy.stats._continuous_distns.expon_gen object at 0x0000015138120350>,"[0.6761145800501458, 2]","[0.7565653350310361, 35]","[0.5564958435768157, 69]","[0.6950833333333333, 69]","[0.7703788748564867, 25]","[0.7212182687598628, 14]","[0.8703679833596352, 69]","[0.716907596371882, 6]","[0.7337910447761193, 64]","[0.6494480401987319, 51]","[0.9166927654243658, 63]","[0.5602741817449851, 3]"
<scipy.stats._continuous_distns.chi2_gen object at 0x00000151381129D0>,"[0.6763770930764142, 2]","[0.7611013846888429, 46]","[0.5568532803450144, 69]","[0.7001666666666667, 66]","[0.7887485648679678, 40]","[0.722259822060184, 14]","[0.8705970694646895, 69]","[0.7381660997732425, 6]","[0.7365335820895523, 66]","[0.650278170965237, 49]","[0.9193026411942791, 67]","[0.5513837026220666, 2]"
<scipy.stats._continuous_distns.gamma_gen object at 0x0000015137E1AF90>,"[0.6764974884502787, 2]","[0.7610814897342033, 41]","[0.5571775126046918, 69]","[0.7001666666666667, 66]","[0.7847301951779564, 26]","[0.722259822060184, 14]","[0.8697028104709015, 68]","[0.7381660997732425, 6]","[0.7359141791044777, 69]","[0.6500094710534449, 49]","[0.9193026411942791, 67]","[0.5606840129167187, 3]"
<scipy.stats._continuous_distns.weibull_min_gen object at 0x00000151381236D0>,"[0.6760776663741966, 2]","[0.7596590004774788, 44]","[0.5578806669027877, 68]","[0.7016388888888889, 68]","[0.7898966704936855, 26]","[0.7036376992980354, 6]","[0.870827360229367, 68]","[0.7414965986394557, 4]","[0.7357313432835821, 68]","[0.6504375532452259, 41]","[0.9186240734941016, 68]","[0.5558884338052292, 3]"
<scipy.stats._continuous_distns.lognorm_gen object at 0x0000015138172050>,"[0.6758756349862142, 2]","[0.7599375298424319, 45]","[0.5563024761448393, 69]","[0.6989444444444444, 69]","[0.791044776119403, 25]","[0.7219392787179627, 14]","[0.8712933627270281, 69]","[0.7385912698412699, 6]","[0.7358059701492536, 63]","[0.6508622192610126, 40]","[0.9206075790792358, 64]","[0.5624772908325756, 2]"


In [6]:
df.to_csv('out.csv')

In [7]:
literature_dataset_paths = [
    "literature/ALOI/ALOI_withoutdupl_norm.arff",
    "literature/Glass/Glass_withoutdupl_norm.arff",
    "literature/Ionosphere/Ionosphere_withoutdupl_norm.arff",
    "literature/KDDCup99/KDDCup99_withoutdupl_norm_idf.arff",
    "literature/Lymphography/Lymphography_withoutdupl_norm_idf.arff",
    "literature/PenDigits/PenDigits_withoutdupl_norm_v10.arff",
    "literature/Shuttle/Shuttle_withoutdupl_norm_v10.arff",
    "literature/Waveform/Waveform_withoutdupl_norm_v10.arff",
    "literature/WBC/WBC_withoutdupl_norm_v10.arff",
    "literature/WDBC/WDBC_withoutdupl_norm_v10.arff",
    "literature/WPBC/WPBC_withoutdupl_norm.arff"
]

positively_skewed_distributions = [
    stats.expon,
    stats.chi2,
    stats.gamma,
    stats.weibull_min,
    stats.lognorm,
    stats.invgauss,
    stats.rayleigh,
    stats.wald,
    stats.pareto,
    stats.levy,
    stats.nakagami,
    stats.logistic,
    stats.powerlaw,
    stats.skewnorm
]



In [ ]:
dict2 = {}
for z in literature_dataset_paths:
    print(z)
    dict2[z] = {}
    for i in positively_skewed_distributions:
        print(i)
        holder = ParametricMethod(z,1,distribution=i)
        maxVal, indexMax = holder.generateOutput()
        dict2[z][i] = [maxVal,indexMax]

df = pd.DataFrame.from_dict(dict2)

literature/ALOI/ALOI_withoutdupl_norm.arff
0.7427896248395957 3


In [ ]:
df.head()

In [ ]:
df.to_csv('literature.csv')